# Add Attendance Points Script
**Ben Paulson -- 9/12/2023**<br>

Given a specific attendance-points form (same format), can parse and add the points to each student that filled out the form. Warnings will be provided for students that are not already included in the form or if there was a typo.<br>

A format to follow is the following W2 attendance form from 2023: https://forms.office.com/Pages/DesignPageV2.aspx?prevorigin=Marketing&origin=NeoPortalPage&subpage=design&id=rM5GQNP9yUasgLfEpJurcGAyFplwhXJCtqB2wsxmGVlUNERLT0g1N0IzOU9aVk1INjE5S0w5VjBRQS4u

# Part 1: Import Statements

In [9]:
# Able to parse excel files using pandas
!pip3 install openpyxl
!pip3 install pandas
!pip3 install numpy 


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import os
# import openpyxl

# Part 2: Parsing the Provided CSV Files
Files in `to_parse` folder will be analyzed and points will be added. Once the sheet is analyzed, will be deleted to save AWS storage costs

In [11]:
attendance_df = {} # {file_name: df}

# Read through all files in the `to_parse` folder
directory_name = 'C:/Users/platert/VS Code/maic-content/to_parse'
for file in os.listdir(directory_name):
    if file.endswith('.xlsx'):
        file_name = directory_name + '/' + file
        attendance_df[file_name] = pd.read_excel(file_name)

In [12]:
website_data = pd.read_csv('C:/Users/platert/VS Code/maic-content/data/points/user_data_8.csv')

In [13]:
pd.notna(website_data.loc[100, 'Awards'])

True

In [14]:
# Create the member_list.csv
# df = website_data
# df['First Name'] = df['User'].str.split(' ').str[0]
# df['Last Name'] = df['User'].str.split(' ').str[1]
# df = df[['First Name', 'Last Name']]

# # Create a new column called 'email'
# # Email is last_name + first letter of first name + @msoe.edu (all lowercase)
# df['email'] = df['Last Name'] + df['First Name'].str[0] + '@msoe.edu'
# df['email'] = df['email'].str.lower()

# df.to_csv('member_list.csv')

In [15]:
for file_name,df in zip(attendance_df.keys(), attendance_df.values()):
    attendance_names = df['First and Last Name'].tolist()
    
    # Print a warning for each of the attendance_names which are not in the 'User' column of the website_data df
    ADD_MISSING_USERS = True
    ADD_BADGE = None
    for name in attendance_names:
        name = name.strip()
        name = name.title()

        # Create the list website_data['User'].tolist() and apply .strip() and .title() to each element
        name_list = website_data['User'].apply(lambda x: x.strip().title()).tolist()
        if name not in name_list:
            print(f'WARNING: {name} is not in the website_data df')
            
            if ADD_MISSING_USERS:
                website_data = pd.concat([website_data, pd.DataFrame({'User': [name], 'Description': ['test description'], 'Awards':[''], 'All-Time Points': [0], 'Current Points': [0]})], ignore_index=True)

        # If name in list and ADD_BADGE isn't None, add the badge to their 'Awards' column
        if ADD_BADGE:
            index = name_list.index(name)
            # If they already have a badge, add the new badge w/ '|' at beginning
            if pd.notna(website_data.loc[index, 'Awards']):
                website_data.loc[index, 'Awards'] += f'|{ADD_BADGE}'
            # If they don't have a badge, add the new badge w/o '|' at beginning
            else:
                website_data.loc[index, 'Awards'] = ADD_BADGE
            

    # Filter out all records in the website_data df where the 'User' column isn't in the attendance_names list
    filtered_website_data = website_data[website_data['User'].isin(attendance_names)]

    # For all the records that are in filtred_website_data, add a 1 to the 'All-Time Points' and 'Current Points' columns in the website_data df
    for index, row in filtered_website_data.iterrows():
        website_data.loc[index, 'All-Time Points'] += 1
        website_data.loc[index, 'Current Points'] += 1
    
    # Save the updated website_data df to a csv file
    # website_data.to_csv('./user_data_2.csv', index=False)

    # Delete the file once finished parsing
    os.remove(file_name)

In [16]:
# If anyone in website_data has 0 All-Time Points and 0 Current Points (at the same time), make sure they are at least given 1 point
website_data.loc[(website_data['All-Time Points'] == 0) & (website_data['Current Points'] == 0), 'All-Time Points'] = 1
website_data.to_csv('C:/Users/platert/VS Code/maic-content/data/points/new_user_data.csv', index=False)

# Part 3: Raffle Point Deductions
Given a raffle sign-up form (same format as the Spring Break Celebration Raffle 2026), processes point spending:
- Each attendee receives **(points_spent + 1)** roulette stickers
- We deduct **points_spent** from their `Current Points` in `user_data.csv`
- If someone submits 0, they get 1 free roulette sticker with no deduction

In [3]:
# ── Config ──────────────────────────────────────────────────────────────────
RAFFLE_XLSX  = r'C:/Users/storoeb/Downloads/MAIC_ Spring Break Celebration Raffle 2026(1-43).xlsx'
USER_DATA_CSV = r'C:/Users/storoeb/OneDrive - Milwaukee School of Engineering/Desktop/ai club/e-board/website/maic-content/data/points/user_data.csv'
OUTPUT_CSV   = USER_DATA_CSV   # overwrite in place; change path to save a separate file
# ────────────────────────────────────────────────────────────────────────────

raffle_df = pd.read_excel(RAFFLE_XLSX)
user_data  = pd.read_csv(USER_DATA_CSV)

NAME_COL   = 'First and Last Name'
POINTS_COL = 'How Many Points Would You Like to Spend?'

print(f'Loaded {len(raffle_df)} raffle entries.')
print(f'Loaded {len(user_data)} users from user_data.csv.')
raffle_df[[NAME_COL, POINTS_COL]].head(10)

Loaded 43 raffle entries.
Loaded 517 users from user_data.csv.


,First and Last Name,How Many Points Would You Like to Spend?
0,Adrian Manchado,1
1,Michael Link,0
2,Mikhail Filippov,3
3,Oliver Grudzinski,0
4,Adam Swedlund,0
5,Patrick Samundsen,0
6,Alexis Jamiola,0
7,Jacob Newberry,0
8,Hayden Park,0
9,Madison Mina,25


In [4]:
for _, row in raffle_df.iterrows():
    name          = str(row[NAME_COL]).strip().title()
    points_to_spend = int(row[POINTS_COL]) if pd.notna(row[POINTS_COL]) else 0
    roulette_stickers = points_to_spend + 1

    name_list = user_data['User'].apply(lambda x: str(x).strip().title()).tolist()

    if name not in name_list:
        print(f'WARNING: {name} not found in user_data.csv '
              f'→ {roulette_stickers} roulette sticker(s), no deduction applied')
        continue

    idx         = name_list.index(name)
    current_pts = int(user_data.loc[idx, 'Current Points'])

    if points_to_spend > current_pts:
        print(f'WARNING: {name} wants to spend {points_to_spend} pts '
              f'but only has {current_pts}. Capping spend at {current_pts}.')
        points_to_spend   = current_pts
        roulette_stickers = points_to_spend + 1

    user_data.loc[idx, 'Current Points'] -= points_to_spend
    remaining = int(user_data.loc[idx, 'Current Points'])
    print(f'{name}: -{points_to_spend} pts → {roulette_stickers} sticker(s)  |  remaining Current Points: {remaining}')

print('\n✓ Done processing raffle entries.')

Adrian Manchado: -1 pts → 2 sticker(s)  |  remaining Current Points: 30
Michael Link: -0 pts → 1 sticker(s)  |  remaining Current Points: 20
Mikhail Filippov: -3 pts → 4 sticker(s)  |  remaining Current Points: 14
Oliver Grudzinski: -0 pts → 1 sticker(s)  |  remaining Current Points: 35
Adam Swedlund: -0 pts → 1 sticker(s)  |  remaining Current Points: 24
Patrick Samundsen: -0 pts → 1 sticker(s)  |  remaining Current Points: 0
Jacob Newberry: -0 pts → 1 sticker(s)  |  remaining Current Points: 21
Hayden Park: -0 pts → 1 sticker(s)  |  remaining Current Points: 16
Madison Mina: -7 pts → 8 sticker(s)  |  remaining Current Points: 0
Isabella Ruiz: -2 pts → 3 sticker(s)  |  remaining Current Points: 0
Paxson Gottschalk: -0 pts → 1 sticker(s)  |  remaining Current Points: 4
Michael Wood: -0 pts → 1 sticker(s)  |  remaining Current Points: 59
Vlad Wilson: -0 pts → 1 sticker(s)  |  remaining Current Points: 30
Owen Pacetti: -11 pts → 12 sticker(s)  |  remaining Current Points: 0
Audrey Chapma

In [5]:
user_data.to_csv(OUTPUT_CSV, index=False)
print(f'Saved updated user data to:\n  {OUTPUT_CSV}')

Saved updated user data to:
  C:/Users/storoeb/OneDrive - Milwaukee School of Engineering/Desktop/ai club/e-board/website/maic-content/data/points/user_data.csv


# Part 4: Add Attendance Points (direct file paths)
Use when you have specific `.xlsx` attendance files rather than dropping them into a folder.
List the files in `ATTENDANCE_FILES` and run the two cells below — each person gets **+1** to both `All-Time Points` and `Current Points`.

In [6]:
# ── Config ──────────────────────────────────────────────────────────────────
ATTENDANCE_FILES = [
    r'C:/Users/storoeb/Downloads/MAIC_ Nigel Nelson @ NVIDIA Speaker Event - 4_9_2026(1-35).xlsx',
    r'C:/Users/storoeb/Downloads/MAIC_ Speaker Event w_ Mayo Clinic (4_23_26)(1-34).xlsx',
]
USER_DATA_CSV = r'C:/Users/storoeb/OneDrive - Milwaukee School of Engineering/Desktop/ai club/e-board/website/maic-content/data/points/user_data.csv'
OUTPUT_CSV    = USER_DATA_CSV   # overwrite in place
ADD_MISSING_USERS = True        # set False to skip unknown names instead of adding them
ADD_BADGE         = None        # set to a badge string to award it to all attendees
# ────────────────────────────────────────────────────────────────────────────

attendance_df = {f: pd.read_excel(f) for f in ATTENDANCE_FILES}
website_data  = pd.read_csv(USER_DATA_CSV)

print(f'Loaded {len(website_data)} users from user_data.csv.')
for f, df in attendance_df.items():
    print(f'  {f.split("/")[-1]}  →  {len(df)} entries')

Loaded 517 users from user_data.csv.
  MAIC_ Nigel Nelson @ NVIDIA Speaker Event - 4_9_2026(1-35).xlsx  →  35 entries
  MAIC_ Speaker Event w_ Mayo Clinic (4_23_26)(1-34).xlsx  →  34 entries


In [7]:
for file_name, df in attendance_df.items():
    event_label = file_name.split('/')[-1]
    print(f'\n── {event_label} ──')
    attendance_names = df['First and Last Name'].tolist()

    for name in attendance_names:
        name = str(name).strip().title()
        name_list = website_data['User'].apply(lambda x: str(x).strip().title()).tolist()

        if name not in name_list:
            print(f'  WARNING: {name} not found in user_data.csv')
            if ADD_MISSING_USERS:
                website_data = pd.concat(
                    [website_data, pd.DataFrame({
                        'User': [name], 'Description': ['test description'],
                        'Awards': [''], 'All-Time Points': [0], 'Current Points': [0],
                        'First Name': [name.split()[0]], 'Last Name': [name.split()[-1]]
                    })],
                    ignore_index=True
                )
                name_list = website_data['User'].apply(lambda x: str(x).strip().title()).tolist()

        if name in name_list:
            idx = name_list.index(name)
            if ADD_BADGE:
                if pd.notna(website_data.loc[idx, 'Awards']) and website_data.loc[idx, 'Awards'] != '':
                    website_data.loc[idx, 'Awards'] += f'|{ADD_BADGE}'
                else:
                    website_data.loc[idx, 'Awards'] = ADD_BADGE
            website_data.loc[idx, 'All-Time Points'] += 1
            website_data.loc[idx, 'Current Points']  += 1
            print(f'  +1  {name}  (All-Time: {int(website_data.loc[idx, "All-Time Points"])}, Current: {int(website_data.loc[idx, "Current Points"])})')

print('\n✓ Done adding attendance points.')


── MAIC_ Nigel Nelson @ NVIDIA Speaker Event - 4_9_2026(1-35).xlsx ──
  +1  Josiah Mathews  (All-Time: 17, Current: 16)
  +1  Madison Mina  (All-Time: 16, Current: 1)
  +1  Adrian Manchado  (All-Time: 34, Current: 31)
  +1  Kassidy Fonk  (All-Time: 14, Current: 1)
  +1  Anthony Carrera  (All-Time: 19, Current: 1)
  +1  Alexander Kruschka  (All-Time: 44, Current: 36)
  +1  Julia Thorne  (All-Time: 4, Current: 3)
  +1  Julian Abelarde  (All-Time: 27, Current: 1)
  +1  John Gillen  (All-Time: 27, Current: 22)
  +1  Morgan Redrup  (All-Time: 18, Current: 11)
  +1  Will Dalebroux  (All-Time: 16, Current: 6)
  +1  Trevor Hancock  (All-Time: 15, Current: 7)
  +1  Audrey Chapman  (All-Time: 51, Current: 41)
  +1  Adam Swedlund  (All-Time: 25, Current: 25)
  +1  Alec Weinbender  (All-Time: 49, Current: 9)
  +1  Ava Latoria  (All-Time: 37, Current: 37)
  +1  Brian Schroedl  (All-Time: 5, Current: 5)
  +1  Brody Sherrill  (All-Time: 11, Current: 11)
  +1  Austin Koske  (All-Time: 76, Current: 36

In [8]:
website_data.to_csv(OUTPUT_CSV, index=False)
print(f'Saved updated user data to:\n  {OUTPUT_CSV}')

Saved updated user data to:
  C:/Users/storoeb/OneDrive - Milwaukee School of Engineering/Desktop/ai club/e-board/website/maic-content/data/points/user_data.csv
